In [5]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
import joblib
import ta  # A nova biblioteca de Análise Técnica
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [6]:
# Definimos Todas Variáveis
tickers = {
    'Bovespa': '^BVSP',        
    'Dolar': 'BRL=X',        
    'SP500': '^GSPC',        
    'Shanghai': '000001.SS', # Bolsa da China
    'Petroleo': 'BZ=F',      
    'Minerio': 'TIO=F',      
    'Ouro': 'GC=F'    
}

In [7]:
print("📥 Baixando cotações...")
df_precos = yf.download(list(tickers.values()), period='3y')['Close']
df_precos.rename(columns={v: k for k, v in tickers.items()}, inplace=True)
df_precos.ffill(inplace=True)
df_precos.dropna(inplace=True)

📥 Baixando cotações...


[*********************100%***********************]  7 of 7 completed


In [8]:
# A. Calculamos os retornos de tudo
for col in df_precos.columns:
    df_features[col] = df_precos[col].pct_change()

NameError: name 'df_features' is not defined

In [ ]:
# B. Injetamos o IFR / RSI (Momento)
df_features['RSI'] = ta.momentum.RSIIndicator(df_precos['Bovespa'], window=14).rsi()

In [ ]:
# C. Injetamos a Tendência (Distância para a Média Móvel Simples de 15 dias)
sma_15 = ta.trend.SMAIndicator(df_precos['Bovespa'], window=15).sma_indicator()
df_features['Distancia_SMA15'] = (df_precos['Bovespa'] / sma_15) - 1

In [ ]:
# D. Injetamos a Volatilidade (Largura das Bandas de Bollinger)
df_features['Bollinger_Width'] = ta.volatility.BollingerBands(df_precos['Bovespa'], window=20).bollinger_wband()

In [ ]:
# Ao calcular médias de 20 dias, os primeiros 20 dias ficam vazios (NaN). Vamos limpar tudo:
df_features.dropna(inplace=True)

In [ ]:
time_steps = 7
colunas_features = df_features.columns

In [ ]:
def classificar_retorno(retorno):
    if retorno <= -0.01: return 0
    elif -0.01 < retorno <= -0.002: return 1
    elif -0.002 < retorno < 0.002: return 2
    elif 0.002 <= retorno < 0.01: return 3
    else: return 4

nomes_classes = {
    0: "📉 Baixa Forte (menor que -1.0%)",
    1: "↘️ Leve Baixa (-1.0% a -0.2%)",
    2: "➖ Neutro (-0.2% a +0.2%)",
    3: "↗️ Leve Alta (+0.2% a +1.0%)",
    4: "📈 Alta Forte (maior que +1.0%)"
}

In [ ]:
def createDatasetClassificacao(dataset, time_steps):
    X, y = [], []
    idx_bovespa = list(dataset.columns).index('Bovespa')
    data_array = dataset.values
    
    for i in range(len(data_array) - time_steps):
        X.append(data_array[i:(i + time_steps)])
        retorno_futuro = data_array[i + time_steps, idx_bovespa]
        y.append(classificar_retorno(retorno_futuro))
        
    return np.array(X), np.array(y)

X, y = createDatasetClassificacao(df_features, time_steps)
y_encoded = to_categorical(y, num_classes=5)

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y_encoded[:split], y_encoded[split:]

In [ ]:
print("📏 Normalizando as Variáveis...")
scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_test_scaled = scaler_X.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

In [ ]:
print("🧠 Treinando Rede Neural...")
modelo_gru = Sequential([
    GRU(50, return_sequences=True, input_shape=(time_steps, len(colunas_features))),
    Dropout(0.2),
    GRU(50, return_sequences=False),
    Dropout(0.2),
    Dense(25, activation='relu'),
    Dense(5, activation='softmax')
])

modelo_gru.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
modelo_gru.fit(X_train_scaled, y_train, epochs=60, batch_size=32, validation_split=0.1, verbose=0)

if not os.path.exists('modelos'): os.makedirs('modelos')
modelo_gru.save_weights("modelos/modelo_gru_classificador.weights.h5", overwrite=True)
joblib.dump(scaler_X, 'modelos/scaler_X_classificador.pkl')

In [ ]:
loss, acuracia = modelo_gru.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\n--- 📊 RESULTADOS DO TREINO (CLASSIFICAÇÃO COM TA) ---")
print(f"Acurácia Global do Modelo: {acuracia * 100:.2f}%")

In [ ]:
ultimos_dias = df_features.tail(time_steps).values
X_futuro_scaled = scaler_X.transform(ultimos_dias.reshape(-1, ultimos_dias.shape[-1])).reshape(1, time_steps, len(colunas_features))

probabilidades = modelo_gru.predict(X_futuro_scaled, verbose=0)[0]
classe_vencedora = np.argmax(probabilidades)

print("\n🚀 PREVISÃO PARA O PRÓXIMO DIA ÚTIL 🚀")
print(f"Cenário mais provável: {nomes_classes[classe_vencedora]}")
print("\nRaio-X de Probabilidades:")
for i in range(5):
    print(f"{nomes_classes[i]}: {probabilidades[i]*100:.2f}%")